In [3]:
# Accept parameters passed from orchestration notebook via dbutils.notebook.run()
# These simulate DAB variables in the bundle deployment

try:
    # Get parameters from dbutils.widgets (passed by dbutils.notebook.run)
    catalog_name = dbutils.widgets.get("catalog_name")
    schema_prefix = dbutils.widgets.get("schema_prefix")
    print(f"Using parameters from orchestration:")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")
except Exception:
    # Fallback to default values if not called from orchestration
    catalog_name = "dev_catalog"
    schema_prefix = "slv_cdm_hrs"
    print(f"Using default values (not called from orchestration):")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")

Using default values (not called from orchestration):
  catalog_name: dev_catalog
  schema_prefix: slv_cdm_hrs


This notebook is used to load and test the HRS Cohort table.  It extracts distinct HACOHORT values from the RAND longitudinal data and populate the cohort table.

**Purpose:** Load the HRS Cohort reference table.

**Source Table:** `dev_catalog.brz_raw_hrs.randhrs1992_2022v1`  
**Target Table:** `dev_catalog.slv_cdm_hrs.cohort`
**Load Script:** `../../sql/dml/load_hrs_cohort_data.sql`
**Validation Script:** `../../sql/validataion/verify_hrs_cohort_data.sql`

**Process:**
1. Clear/truncate the HRS COHORT table .
2. Extract distinct HACOHORT values from source data and load the HRS COHORT table.
3. Validate the table data. 
4. Display summary stats.

In [ ]:
# -----------------------------------------------------------------------------
# Initialize Notebook Configuration
# ----------------------------------------------------------------------------
# Variables are received from the first cell (either from orchestration or defaults)

dbutils.widgets.dropdown(
    "truncate_table",
    "true",
    ["true", "false"]
)

TRUNCATE_TABLE = dbutils.widgets.get("truncate_table").lower() == "true"

# Build target table name from parameters
TARGET_TABLE = f"{catalog_name}.{schema_prefix}.dim_cohort"

LOAD_SQL = "../../sql/dml/load_hrs_cohort_data.sql"

VALIDATION_SQL = "../../sql/validation/validate_hrs_cohort_data.sql"

SOURCE_TABLE = "dev_catalog.brz_raw_hrs.randhrs1992_2022v1"

In [ ]:
# Step 1:
# Clear existing cohort data if needed (use with caution)
# Uncomment the line below to truncate the table before loading 

if TRUNCATE_TABLE:
    print("======================================================")
    print("Step 1 - TRUNCATE")
    print("======================================================")

    try:
        spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        print("✓ Completed")
    except Exception as e:
        print(f"❌ TRUNCATE failed: {e}")
        raise

else:
    print("Table not found.  Skipping table truncation.")

In [ ]:
# Step 2
# Load distinct data to the TARGET_TABLE

from pathlib import Path
import re

print("======================================================")
print("Step 2 - LOAD DATA")
print(catalog_name)
print(schema_prefix)
print("======================================================")
try:
    # Read SQL file
    sql_path = Path(LOAD_SQL)
    sql_text = sql_path.read_text()

    print(sql_path)

    # Replace SQL parameters with actual values
    #sql_text = sql_text.replace(':catalog_name', f"'{catalog_name}'")
    #sql_text = sql_text.replace(':schema_prefix', f"'{schema_prefix}'")

    sql_text = sql_text.replace('dev_catalog', f"{catalog_name}")

    # Replace the IDENTIFIER(CONCAT(...)) pattern with direct table reference
    # This fixes the RDD_BASED error that occurs with parameter substitution.  
    # Replaces the global variables with the IDENTIFIER(CONCAT(...)) with the TARGET_TABLE variable.
    #pattern = r'IDENTIFIER\(\s*CONCAT\([^)]*\)\s*\)'
    #sql_text = re.sub(pattern, TARGET_TABLE, sql_text)
    
    print('text = ', sql_text)
    
    # Split and execute statements
    statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"  Executing statement {i}/{len(statements)}")
        spark.sql(stmt)
    
    print("✓ Completed")
except Exception as e:
    print(f"❌ Load failed: {e}")
    raise


In [ ]:
# Step 3: Verify the TARGET_TABLE.
from pathlib import Path

print("======================================================")
print("Step 3 - Validation")
print("======================================================")
try:
    # Read SQL file
    sql_path = Path(VALIDATION_SQL)
    sql_text = sql_path.read_text()
    
    # Split and execute statements with parameter binding
    statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]
    
    for i, stmt in enumerate(statements, 1):
        print(f"  Executing statement {i}/{len(statements)}")
        result = spark.sql(stmt, args={"catalog_name": catalog_name, "schema_prefix": schema_prefix})
        display(result)
    
    print("✓ Completed")
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise

In [ ]:
# Step 4 - Display summary statistics

source_count = spark.sql(f"""
    SELECT COUNT(DISTINCT HACOHORT) as distinct_cohorts
    FROM {SOURCE_TABLE}
    WHERE HACOHORT IS NOT NULL
""").collect()[0][0]

target_count = spark.sql(f"""
    SELECT COUNT(*) as cohort_count
    FROM {TARGET_TABLE}
""").collect()[0][0]

print("=" * 60)
print("HRS COHORT DATA LOAD SUMMARY")
print("=" * 60)
print(f"Distinct HACOHORT values in source: {source_count}")
print(f"Total records in HRS cohort table:      {target_count}")
print("=" * 60)

if source_count == target_count:
    print("✓ SUCCESS: All distinct HRS cohorts loaded")
else:
    print(f"⚠ WARNING: Mismatch detected. Please review.")